# AMST Shape Descriptor v16 - MPEG-7 CE-Shape-1 Part B (FIXED)

**Adaptive Multi-Scale Topological Shape Descriptor** with filter-bank covariance, per-component normalization, and Fisher-guided feature selection.

### Key fixes in v16:
- **C3 (SPD)** replaced with filter-bank covariance (20 base features × 4 scales → 210-d)
- **C2 (Topological)** fixed: `max_edge_length=50.0` + full exception handling
- **Per-component normalization** inside CV loop (prevents C3 from poisoning)
- **Fisher scores for feature selection** (SelectKBest k=300) — no multiplicative weighting
- **Direct SVM-RBF** (no stacking, no grid search for AMST)


In [ ]:
import subprocess, sys, importlib, warnings, os
warnings.filterwarnings('ignore')
deps = ['numpy', 'scipy', 'scikit-learn', 'matplotlib', 'seaborn',
        'opencv-python', 'pillow', 'xgboost', 'tensorflow', 'gdown', 'pywavelets']
for d in deps:
    try:
        mn = d.replace('-','_').replace('opencv-python','cv2').replace('pillow','PIL').replace('scikit-learn','sklearn')
        importlib.import_module(mn)
    except ImportError:
        print(f"Installing {d}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', d, '-q'])
print("All dependencies available.")


In [ ]:
import os, sys, json, time, warnings, pickle, copy, itertools, random, hashlib, gzip
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from PIL import Image
from io import BytesIO
import zipfile, tarfile
from pathlib import Path
from collections import Counter, defaultdict, OrderedDict
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             precision_recall_curve, average_precision_score)
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from scipy.ndimage import gaussian_filter, sobel, label, binary_fill_holes
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.stats import ttest_rel, ttest_ind
from scipy.signal import find_peaks
from scipy.spatial import ConvexHull
from skimage import measure, feature, morphology, transform, filters, segmentation, color, exposure, draw
from skimage.feature import hog, local_binary_pattern
from skimage.measure import find_contours, approximate_polygon
from skimage.filters import gabor
import pywt
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model

# GUDHI for topological persistence
try:
    import gudhi as gd
    HAS_GUDHI = True
except ImportError:
    HAS_GUDHI = False
    print("WARNING: GUDHI not installed. C2 features will return zeros. Install with: pip install gudhi")

np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)
print("All imports successful.")


In [ ]:
DATA_DIR = 'mpeg7_data'
os.makedirs(DATA_DIR, exist_ok=True)

# Check if already extracted
if os.path.exists(os.path.join(DATA_DIR, 'device')) and len(os.listdir(os.path.join(DATA_DIR, 'device'))) > 10:
    print(f"Dataset already extracted in {DATA_DIR}/")
else:
    # Try to find any zip in current directory or DATA_DIR
    zip_candidates = []
    for p in ['.', DATA_DIR]:
        if os.path.isdir(p):
            for f in os.listdir(p):
                if 'mpeg' in f.lower() or 'shape' in f.lower() or 'CE' in f:
                    zip_candidates.append(os.path.join(p, f))
    if zip_candidates:
        zpath = zip_candidates[0]
        print(f"Found archive: {zpath}")
    else:
        # Download from alternative source
        import requests as req
        url = "https://github.com/nalin-singh/MPEG7-Shape-Descriptor-Dataset/raw/master/MPEG7.zip"
        print(f"Downloading from {url}...")
        r = req.get(url, stream=True, timeout=300)
        zpath = os.path.join(DATA_DIR, 'MPEG7.zip')
        with open(zpath, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print("Download complete.")
    with zipfile.ZipFile(zpath, 'r') as zf:
        zf.extractall(DATA_DIR)
    print(f"Extracted to {DATA_DIR}/")

# List classes
classes = sorted([d for d in os.listdir(DATA_DIR)
                  if os.path.isdir(os.path.join(DATA_DIR, d)) and not d.startswith('_') and not d.startswith('.')])
print(f"Found {len(classes)} classes:")
print(classes[:10], '...' if len(classes) > 10 else '')


In [ ]:
# Build image paths and labels
image_paths = []
labels = []
for cls in classes:
    cls_dir = os.path.join(DATA_DIR, cls)
    for fname in sorted(os.listdir(cls_dir)):
        if fname.lower().endswith(('.gif', '.png', '.jpg', '.jpeg', '.bmp')):
            image_paths.append(os.path.join(cls_dir, fname))
            labels.append(cls)

print(f"Total images: {len(image_paths)}")
print(f"Total classes: {len(set(labels))}")
cls_counts = Counter(labels)
for cls, cnt in cls_counts.most_common(5):
    print(f"  {cls}: {cnt} samples")
print("  ...")
for cls, cnt in cls_counts.most_common()[-3:]:
    print(f"  {cls}: {cnt} samples")


In [ ]:
def load_and_preprocess(path, target_size=(128, 128)):
    '''Load image, convert to binary shape.'''
    img = Image.open(path).convert('L')
    img = img.resize(target_size, Image.LANCZOS)
    arr = np.array(img, dtype=np.float32)
    # Binary threshold
    _, bw = cv2.threshold(arr.astype(np.uint8), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    bw = (bw > 127).astype(np.float32)
    return bw

def extract_contour(bw_img):
    '''Extract contour points from binary image.'''
    from skimage import measure
    try:
        contours = measure.find_contours(bw_img, level=0.5)
        if len(contours) == 0:
            return np.zeros((10, 2))
        # Take the longest contour
        contour = max(contours, key=len)
        return contour
    except Exception:
        return np.zeros((10, 2))

def extract_contour_points(bw_img, n_points=200):
    '''Extract and subsample contour points.'''
    contour = extract_contour(bw_img)
    if len(contour) < 10:
        return np.zeros((n_points, 2))
    # Uniform subsampling
    indices = np.linspace(0, len(contour) - 1, n_points, dtype=int)
    return contour[indices]

# Preload all images and contours (cached)
CACHE_FILE = 'preprocessed_cache_v16.npy'
if os.path.exists(CACHE_FILE):
    print("Loading preprocessed cache...")
    cache = np.load(CACHE_FILE, allow_pickle=True).item()
    all_images = cache['images']
    all_contours = cache['contours']
    print(f"Loaded {len(all_images)} images from cache.")
else:
    print("Preprocessing all images...")
    all_images = []
    all_contours = []
    for i, p in enumerate(image_paths):
        if (i+1) % 200 == 0:
            print(f"  [{i+1}/{len(image_paths)}]")
        bw = load_and_preprocess(p)
        contour = extract_contour_points(bw, n_points=200)
        all_images.append(bw)
        all_contours.append(contour)
    np.save(CACHE_FILE, {'images': np.array(all_images), 'contours': np.array(all_contours)})
    print(f"Saved {len(all_images)} images to cache.")

all_images = np.array(all_images)
all_contours = np.array(all_contours)
print(f"Image array shape: {all_images.shape}")
print(f"Contour array shape: {all_contours.shape}")


In [ ]:
# Check if we can generate figures in headless mode
fig, axes = plt.subplots(7, 9, figsize=(15, 12))
unique_classes = sorted(set(labels))
for i, ax in enumerate(axes.flat):
    if i < len(unique_classes):
        cls = unique_classes[i]
        idx = labels.index(cls)
        ax.imshow(all_images[idx], cmap='gray')
        ax.set_title(cls, fontsize=7)
        ax.axis('off')
    else:
        ax.axis('off')
plt.suptitle('MPEG-7 CE-Shape-1 Part B - One Sample per Class (63 classes)', fontsize=14)
plt.tight_layout()
plt.savefig('figure1_dataset_samples.png', dpi=150, bbox_inches='tight')
plt.close()
print("Figure 1 saved: figure1_dataset_samples.png")


In [ ]:
# Show preprocessing pipeline on one sample
idx = 0
img = Image.open(image_paths[idx]).convert('L')
img_orig = np.array(img, dtype=np.float32)
bw = load_and_preprocess(image_paths[idx])
contour = extract_contour(bw)
contour_pts = extract_contour_points(bw, n_points=100)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
steps = [
    ('Original', img_orig, 'gray'),
    ('Grayscale', img_orig / 255.0, 'gray'),
    ('Binary (Otsu)', bw, 'gray'),
    ('Contour', bw, 'gray'),
    ('Subsampled Points (100)', bw, 'gray'),
]
for ax, (title, data, cm) in zip(axes, steps):
    ax.imshow(data, cmap=cm)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
# Add contour overlay
axes[3].plot(contour[:, 1], contour[:, 0], 'r-', linewidth=1)
axes[4].plot(contour_pts[:, 1], contour_pts[:, 0], 'ro', markersize=2)
plt.suptitle('Preprocessing Pipeline', fontsize=14)
plt.tight_layout()
plt.savefig('figure2_preprocessing.png', dpi=150, bbox_inches='tight')
plt.close()
print("Figure 2 saved: figure2_preprocessing.png")


## Baseline Shape Descriptors

We compare against six classical shape descriptors:
1. **HOG** (Histogram of Oriented Gradients) — 1764-d → PCA to 128-d
2. **Zernike Moments** — degree 20, 121-d → PCA to 128-d
3. **Shape Context** — 200 points, 5×12 bins = 60-d
4. **CSS** (Curvature Scale Space) — 40 scales, 40-d
5. **Fourier Descriptors** — 40 coeffs × 2 = 80-d
6. **Wavelet Descriptors** — energy + entropy per subband


In [ ]:
# ─── HOG ───
def extract_hog(bw_img):
    from skimage.feature import hog as hog_feat
    try:
        fv = hog_feat(bw_img, pixels_per_cell=(8,8), cells_per_block=(3,3),
                      orientations=9, feature_vector=True)
        return fv
    except Exception:
        return np.zeros(1764)

# ─── Zernike Moments ───
from skimage.feature import hog as sk_hog  # already imported above

def zernike_moments(bw_img, degree=20):
    from skimage import measure as skmeas
    try:
        moms = skmeas.moments_zernike(bw_img, radius=bw_img.shape[0]//2, degree=degree)
        return moms
    except Exception:
        return np.zeros(121)

# ─── Shape Context ───
def shape_context_descriptor(contour_pts, n_r=5, n_theta=12):
    from skimage.feature import match_descriptors
    try:
        pts = contour_pts.copy()
        if len(pts) < 10:
            return np.zeros(n_r * n_theta)
        # Compute log-polar bins
        cx, cy = pts.mean(axis=0)
        d = pts - np.array([cx, cy])
        r = np.sqrt(d[:,0]**2 + d[:,1]**2) + 1e-10
        theta = np.arctan2(d[:,1], d[:,0])
        r_bins = np.logspace(np.log10(r.min()), np.log10(r.max()), n_r+1)
        theta_bins = np.linspace(-np.pi, np.pi, n_theta+1)
        hist = np.zeros((n_r, n_theta))
        for i in range(len(pts)):
            ri = np.searchsorted(r_bins, r[i]) - 1
            ti = np.searchsorted(theta_bins, theta[i]) - 1
            if 0 <= ri < n_r and 0 <= ti < n_theta:
                hist[ri, ti] += 1
        return hist.flatten() / len(pts)
    except Exception:
        return np.zeros(n_r * n_theta)

# ─── CSS ───
def css_descriptor(bw_img, n_scales=40):
    try:
        contour = extract_contour(bw_img)
        if len(contour) < 10:
            return np.zeros(n_scales)
        # Parametric curve
        t = np.linspace(0, 2*np.pi, len(contour), endpoint=False)
        x = contour[:, 1]; y = contour[:, 0]
        sigma_min = 1; sigma_max = 20
        css = np.zeros(n_scales)
        for i, sigma in enumerate(np.linspace(sigma_min, sigma_max, n_scales)):
            xs = gaussian_filter(x, sigma, mode='wrap')
            ys = gaussian_filter(y, sigma, mode='wrap')
            dx = np.gradient(xs); dy = np.gradient(ys)
            ddx = np.gradient(dx); ddy = np.gradient(dy)
            k = (dx * ddy - dy * ddx) / (dx**2 + dy**2 + 1e-10)**1.5
            # Find zero-crossings
            signs = np.sign(k)
            zc = np.sum(np.abs(np.diff(signs)) > 0)
            css[i] = zc
        return css
    except Exception:
        return np.zeros(n_scales)

# ─── Fourier Descriptors ───
def fourier_descriptor(contour_pts, n_coeffs=40):
    try:
        pts = contour_pts.copy()
        if len(pts) < 10:
            return np.zeros(2 * n_coeffs)
        # Complex representation
        z = pts[:, 1] + 1j * pts[:, 0]
        # Normalize by centroid
        z = z - z.mean()
        fft_coeffs = np.fft.fft(z)
        # Take magnitude of first n_coeffs (excluding DC)
        mags = np.abs(fft_coeffs[1:n_coeffs+1])
        phases = np.angle(fft_coeffs[1:n_coeffs+1])
        return np.concatenate([mags, phases])
    except Exception:
        return np.zeros(2 * n_coeffs)

# ─── Wavelet Descriptors ───
def wavelet_descriptor(bw_img, wavelet='db4', level=4):
    try:
        coeffs = pywt.wavedec2(bw_img, wavelet, level=level)
        feats = []
        for i, c in enumerate(coeffs):
            if i == 0:
                feats.extend([np.mean(c), np.std(c), np.sum(c**2) / c.size])
            else:
                for detail in c:
                    feats.extend([np.mean(detail), np.std(detail), np.sum(detail**2) / detail.size,
                                 -np.sum(detail**2 * np.log(detail**2 + 1e-10))])
        return np.array(feats)
    except Exception:
        return np.zeros(30)

# ─── Wrapper ───
def extract_baselines(bw_img, contour_pts):
    h = extract_hog(bw_img)
    z = zernike_moments(bw_img)
    s = shape_context_descriptor(contour_pts)
    c = css_descriptor(bw_img)
    f = fourier_descriptor(contour_pts)
    w = wavelet_descriptor(bw_img)
    return {'hog': h, 'zernike': z, 'shape_context': s, 'css': c, 'fourier': f, 'wavelet': w}

print("Baseline descriptor functions defined.")


## AMST Descriptor v16 (Proposed)

The **Adaptive Multi-Scale Topological (AMST)** descriptor combines five complementary components:

| Component | Name | Dim | Description |
|-----------|------|-----|-------------|
| **C1** | APCFW+ | 160 | Angular Pairwise Contour Feature Weighted+ |
| **C2** | Topological Persistence | 90 | Persistent homology of contour points (FIXED max_edge_length=50.0) |
| **C3** | SPD (Filter-Bank Covariance) | 210 | Filter-bank covariance on distance transform (REPLACED in v16) |
| **C4** | Deep Features | 128 | MobileNetV2 bottleneck features |
| **C5** | Shape Complexity | 30 | Multi-scale shape complexity measures |

**v16 fixes:**
- C3 uses filter-bank (20 base features × 4 scales) instead of patch-based PCA
- C2 uses max_edge_length=50.0 and catches all exceptions
- Per-component normalization + Fisher feature selection (k=300) + direct SVM-RBF


In [ ]:
# ─── C1: APCFW+ (160-d) ───
def apcfw_plus(bw_img, contour_pts):
    '''Angular Pairwise Contour Feature Weighted+'''
    try:
        pts = contour_pts.copy()
        if len(pts) < 10:
            return np.zeros(160)
        # Centroid
        cx, cy = pts.mean(axis=0)
        rel = pts - np.array([cx, cy])
        r = np.sqrt(rel[:,0]**2 + rel[:,1]**2) + 1e-10
        theta = np.arctan2(rel[:,1], rel[:,0])
        n_bins_angle = 16
        n_bins_dist = 10
        hist = np.zeros((n_bins_dist, n_bins_angle))
        angle_bins = np.linspace(-np.pi, np.pi, n_bins_angle + 1)
        dist_bins = np.linspace(0, np.percentile(r, 95), n_bins_dist + 1)
        for i in range(len(pts)):
            ai = np.searchsorted(angle_bins, theta[i]) - 1
            di = np.searchsorted(dist_bins, r[i]) - 1
            if 0 <= ai < n_bins_angle and 0 <= di < n_bins_dist:
                hist[di, ai] += 1
        fv = hist.flatten() / len(pts)
        # Pad or truncate to 160
        if len(fv) < 160:
            fv = np.pad(fv, (0, 160 - len(fv)))
        else:
            fv = fv[:160]
        return fv
    except Exception:
        return np.zeros(160)

# ─── C2: Topological Persistence (90-d) FIXED ───
def topological_persistence(contour_pts):
    '''Persistent homology of contour points. max_edge_length=50.0'''
    global HAS_GUDHI
    try:
        pts = contour_pts.copy()
        if len(pts) < 10:
            return np.zeros(90)
        if not HAS_GUDHI:
            return np.zeros(90)
        pts = pts.astype(float)
        # Build Rips complex with FIXED max_edge_length
        rc = gd.RipsComplex(points=pts, max_edge_length=50.0)
        st = rc.create_simplex_tree(max_dimension=2)
        st.compute_persistence()
        # Extract H1 persistence
        pairs = st.persistence()
        h1_pairs = [p[1] for p in pairs if p[0] == 1]
        feats = []
        for (b, d) in h1_pairs[:30]:
            feats.extend([b, d, d - b if d != float('inf') else 50.0])
        while len(feats) < 90:
            feats.append(0.0)
        return np.array(feats[:90])
    except Exception:
        return np.zeros(90)

# ─── C3: SPD Filter-Bank Covariance (210-d) REPLACED ───
def spd_descriptor(img):
    '''Filter-bank covariance: 20 base features x 4 scales = 20 features, 210-d SPD.
    Uses distance transform + multi-scale Gaussian derivatives.'''
    try:
        img_f = img.astype(float)
        dist = cv2.distanceTransform((img_f > 0.5).astype(np.uint8), cv2.DIST_L2, 5)
        all_feat = []
        for sigma in [1, 2, 4, 8]:
            g_img = gaussian_filter(img_f, sigma)
            g_dist = gaussian_filter(dist, sigma)
            gx = sobel(g_img, axis=1)
            gy = sobel(g_img, axis=0)
            mag = np.sqrt(gx**2 + gy**2)
            dx = sobel(g_dist, axis=1)
            dy = sobel(g_dist, axis=0)
            dmag = np.sqrt(dx**2 + dy**2)
            lap = cv2.Laplacian((np.clip(g_img, 0, 1) * 255).astype(np.uint8), cv2.CV_32F).flatten() / 255.0
            all_feat.extend([g_img.flatten(), g_dist.flatten(), mag.flatten(), dmag.flatten(), lap])
        F = np.array(all_feat)
        F_c = F - F.mean(axis=1, keepdims=True)
        C = (F_c @ F_c.T) / (F.shape[1] - 1)
        C += 1e-6 * np.eye(C.shape[0])
        eigvals, eigvecs = np.linalg.eigh(C)
        log_C = eigvecs @ np.diag(np.log(np.maximum(eigvals, 1e-10))) @ eigvecs.T
        spd = log_C[np.triu_indices_from(log_C)]
        return np.pad(spd, (0, max(0, 210 - len(spd))), constant_values=0)[:210]
    except Exception:
        return np.zeros(210)

# ─── C4: Deep Features (128-d) ───
def _get_mobilenet_base():
    try:
        base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3), pooling='avg')
        return Model(inputs=base.input, outputs=base.output)
    except Exception as e:
        print(f"  MobileNetV2 load failed: {e}")
        return None

_mobilenet_base = None

def deep_features(img, model=None):
    global _mobilenet_base
    try:
        if model is None:
            if _mobilenet_base is None:
                _mobilenet_base = _get_mobilenet_base()
            model = _mobilenet_base
        if model is None:
            return np.zeros(128)
        rgb = np.stack([img * 255.0] * 3, axis=-1).astype(np.uint8)
        rgb = np.expand_dims(rgb, 0)
        feats = model.predict(rgb, verbose=0).flatten()
        return feats[:128]
    except Exception:
        return np.zeros(128)

# ─── C5: Shape Complexity (30-d) ───
def shape_complexity(bw_img, contour_pts):
    try:
        feats = []
        # 1. Area and perimeter
        area = np.sum(bw_img)
        perimeter = len(contour_pts)
        feats.extend([area / (128*128), perimeter / (128*4)])
        # 2. Compactness
        compactness = (perimeter**2) / (4 * np.pi * area + 1e-10)
        feats.append(np.log(compactness + 1e-10))
        # 3. Convexity
        if len(contour_pts) >= 3:
            hull = ConvexHull(contour_pts)
            hull_area = hull.volume
            feats.append(area / (hull_area + 1e-10))
        else:
            feats.append(0.0)
        # 4. Eccentricity
        moments = cv2.moments(bw_img.astype(np.uint8))
        if moments['mu20'] + moments['mu02'] > 0:
            ecc = ((moments['mu20'] - moments['mu02'])**2 + 4*moments['mu11']**2) / (moments['mu20'] + moments['mu02'])**2
            feats.append(ecc)
        else:
            feats.append(0.0)
        # 5. Multi-scale complexity via downsampling
        for scale in [64, 32, 16, 8, 4]:
            small = cv2.resize(bw_img.astype(np.uint8), (scale, scale), interpolation=cv2.INTER_NEAREST)
            feats.append(np.sum(small) / (scale**2))
        # Fill to 30
        while len(feats) < 30:
            feats.append(0.0)
        return np.array(feats[:30])
    except Exception:
        return np.zeros(30)

# ─── AMSTDescriptor class ───
class AMSTDescriptor:
    def __init__(self):
        self._mobilenet_model = None

    def extract_raw(self, bw_img, contour_pts, precomputed_deep=None):
        c1 = apcfw_plus(bw_img, contour_pts)
        c2 = topological_persistence(contour_pts)
        c3 = spd_descriptor(bw_img)
        if precomputed_deep is not None:
            c4 = precomputed_deep
        else:
            c4 = deep_features(bw_img, self._mobilenet_model)
        c5 = shape_complexity(bw_img, contour_pts)
        return np.concatenate([c1, c2, c3, c4, c5])

amst_desc = AMSTDescriptor()
print("AMST descriptor v16 defined with all fixes.")
print(f"  C1 (APCFW+): 160-d")
print(f"  C2 (Topological): 90-d")
print(f"  C3 (SPD Filter-Bank): 210-d")
print(f"  C4 (Deep): 128-d")
print(f"  C5 (Complexity): 30-d")
print(f"  Total: 618-d")


In [ ]:
# APCFW+ analysis on a few samples
sample_idx = [0, 100, 200, 300]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, sidx in enumerate(sample_idx):
    if sidx >= len(all_images):
        continue
    fv = apcfw_plus(all_images[sidx], all_contours[sidx])
    axes[i].bar(range(len(fv)), fv, width=1)
    axes[i].set_title(f'{labels[sidx]}', fontsize=8)
    axes[i].set_xlabel('Feature index')
    axes[i].set_ylabel('Value')
plt.suptitle('C1: APCFW+ Feature Responses', fontsize=14)
plt.tight_layout()
plt.savefig('figure3_apcfw_plus.png', dpi=150, bbox_inches='tight')
plt.close()
print("Figure 3 saved: figure3_apcfw_plus.png")


In [ ]:
# Topological persistence diagram on one sample
if HAS_GUDHI:
    idx = 0
    pts = all_contours[idx].astype(float)
    rc = gd.RipsComplex(points=pts, max_edge_length=50.0)
    st = rc.create_simplex_tree(max_dimension=2)
    st.compute_persistence()
    pairs = st.persistence()
    h1_pairs = [(b, d) for (t, (b, d)) in pairs if t == 1 and d != float('inf')]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(all_images[idx], cmap='gray')
    axes[0].plot(pts[:, 1], pts[:, 0], 'r.', markersize=1)
    axes[0].set_title(f'Shape: {labels[idx]}')
    axes[0].axis('off')

    if h1_pairs:
        births = [b for b, d in h1_pairs]
        deaths = [d for b, d in h1_pairs]
        axes[1].scatter(births, deaths, c='blue', alpha=0.6, s=10)
        axes[1].plot([0, 50], [0, 50], 'r--', alpha=0.5)
    axes[1].set_xlabel('Birth')
    axes[1].set_ylabel('Death')
    axes[1].set_title('H1 Persistence Diagram')
    axes[1].set_xlim(0, 55); axes[1].set_ylim(0, 55)
    plt.suptitle('C2: Topological Persistence Analysis', fontsize=14)
    plt.tight_layout()
    plt.savefig('figure4_topological.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Figure 4 saved: figure4_topological.png")
else:
    print("GUDHI not available, skipping Figure 4.")


In [ ]:
# Extract deep features FIRST (cached)
DEEP_CACHE = 'deep_features_cache_v16.npy'
FEATURES_CACHE = 'features_cache_v16.npy'

if os.path.exists(DEEP_CACHE) and os.path.exists(FEATURES_CACHE):
    print("Loading cached features...")
    deep_feat_array = np.load(DEEP_CACHE)
    cache_data = np.load(FEATURES_CACHE, allow_pickle=True).item()
    baseline_feats = cache_data['baseline']
    amst_raw_feats = cache_data['amst_raw']
    print(f"Loaded deep features: {deep_feat_array.shape}")
    print(f"Loaded baseline features: {[f': {v.shape} ' for f,v in baseline_feats.items()]}")
    print(f"Loaded AMST raw: {amst_raw_feats.shape}")
else:
    print("Extracting deep features (first pass)...")
    _mb = _get_mobilenet_base()
    deep_feat_array = np.zeros((len(all_images), 128))
    for i in range(len(all_images)):
        if (i+1) % 200 == 0:
            print(f"  Deep [{i+1}/{len(all_images)}]")
        deep_feat_array[i] = deep_features(all_images[i], _mb)
    np.save(DEEP_CACHE, deep_feat_array)
    print(f"Deep features saved: {deep_feat_array.shape}")

    print("Extracting all descriptors...")
    baseline_feats = {'hog': [], 'zernike': [], 'shape_context': [], 'css': [], 'fourier': [], 'wavelet': []}
    amst_raw_list = []
    for i in range(len(all_images)):
        if (i+1) % 200 == 0:
            print(f"  Total [{i+1}/{len(all_images)}]")
        bfeats = extract_baselines(all_images[i], all_contours[i])
        for k in baseline_feats:
            baseline_feats[k].append(bfeats[k])
        raw = amst_desc.extract_raw(all_images[i], all_contours[i], precomputed_deep=deep_feat_array[i])
        amst_raw_list.append(raw)

    baseline_feats = {k: np.array(v) for k, v in baseline_feats.items()}
    amst_raw_feats = np.array(amst_raw_list)
    np.save(FEATURES_CACHE, {'baseline': baseline_feats, 'amst_raw': amst_raw_feats})
    print(f"Features saved. AMST raw shape: {amst_raw_feats.shape}")
    for k, v in baseline_feats.items():
        print(f"  {k}: {v.shape}")

# Encode labels
le = LabelEncoder()
y = le.fit_transform(labels)
n_classes = len(le.classes_)
print(f"Labels encoded: {n_classes} classes, {len(y)} samples")


In [ ]:
# ─── PCA reduction for baselines ───
pca_128 = PCA(n_components=128, random_state=42)
baseline_pca = {}
for name, feats in baseline_feats.items():
    try:
        baseline_pca[name] = pca_128.fit_transform(feats)
    except Exception:
        baseline_pca[name] = feats[:, :128] if feats.shape[1] >= 128 else np.pad(feats, ((0,0),(0,128-feats.shape[1])))

# ─── Combined deep baseline ───
baseline_pca['deep'] = deep_feat_array

# ─── Standard 5-fold CV with grid search SVM for baselines ───
def run_5fold_cv_baseline(features, y, name):
    skf = StratifiedKFold(5, shuffle=True, random_state=42)
    scores, all_yt, all_yp = [], [], []
    for train_idx, test_idx in skf.split(features, y):
        X_tr, X_te = features[train_idx], features[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        ss = StandardScaler()
        X_tr = ss.fit_transform(X_tr)
        X_te = ss.transform(X_te)
        gs = GridSearchCV(SVC(kernel='rbf', random_state=42),
                          {'C': [1, 10], 'gamma': ['scale']},
                          cv=3, scoring='accuracy', n_jobs=1)
        gs.fit(X_tr, y_tr)
        yp = gs.predict(X_te)
        scores.append(accuracy_score(y_te, yp))
        all_yt.extend(y_te)
        all_yp.extend(yp)
    return scores, all_yt, all_yp

# ─── 5-fold CV for AMST with per-component norm + Fisher selection + SVM-RBF ───
def run_5fold_cv_amst(X_raw, y):
    component_dims = [160, 90, 210, 128, 30]
    skf = StratifiedKFold(5, shuffle=True, random_state=42)
    scores, all_yt, all_yp = [], [], []
    for train_idx, test_idx in skf.split(X_raw, y):
        X_tr_raw, X_te_raw = X_raw[train_idx], X_raw[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        # Step 1: Per-component normalization (fit on train, apply to both)
        X_tr_norm = np.zeros_like(X_tr_raw)
        X_te_norm = np.zeros_like(X_te_raw)
        start = 0
        for dim in component_dims:
            end = start + dim
            blk_tr = X_tr_raw[:, start:end]
            blk_te = X_te_raw[:, start:end]
            mean = blk_tr.mean(axis=0)
            std = blk_tr.std(axis=0) + 1e-10
            X_tr_norm[:, start:end] = (blk_tr - mean) / std
            X_te_norm[:, start:end] = (blk_te - mean) / std
            start = end

        # Step 2: Fisher score feature SELECTION (not weighting!)
        k = min(300, X_tr_norm.shape[1])
        selector = SelectKBest(f_classif, k=k)
        X_tr_sel = selector.fit_transform(X_tr_norm, y_tr)
        X_te_sel = selector.transform(X_te_norm)

        # Step 3: Standardize
        ss = StandardScaler()
        X_tr = ss.fit_transform(X_tr_sel)
        X_te = ss.transform(X_te_sel)

        # Step 4: Direct SVM (no stacking, no grid search)
        clf = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
        clf.fit(X_tr, y_tr)
        yp = clf.predict(X_te)

        acc = accuracy_score(y_te, yp)
        scores.append(acc)
        all_yt.extend(y_te)
        all_yp.extend(yp)

    mean_acc = np.mean(scores) * 100
    print(f'  AMST (Proposed): {mean_acc:.2f}% +/- {np.std(scores)*100:.2f}%')
    return scores, all_yt, all_yp

# ─── RUN ALL ───
print("Running 5-fold cross-validation...")
print()

all_results = {}
all_predictions = {}

for name in ['hog', 'zernike', 'shape_context', 'css', 'fourier', 'wavelet', 'deep']:
    print(f"  {name}...")
    scores, yt, yp = run_5fold_cv_baseline(baseline_pca[name], y, name)
    all_results[name] = {'scores': scores, 'mean': np.mean(scores)*100, 'std': np.std(scores)*100}
    all_predictions[name] = (yt, yp)
    print(f"    Mean: {all_results[name]['mean']:.2f}% +/- {all_results[name]['std']:.2f}%")

print()
print("  AMST raw (no selection)...")
scores_raw, yt_raw, yp_raw = run_5fold_cv_baseline(amst_raw_feats, y, 'amst_raw')
all_results['amst_raw'] = {'scores': scores_raw, 'mean': np.mean(scores_raw)*100, 'std': np.std(scores_raw)*100}
all_predictions['amst_raw'] = (yt_raw, yp_raw)
print(f"    Mean: {all_results['amst_raw']['mean']:.2f}% +/- {all_results['amst_raw']['std']:.2f}%")

print()
print("  AMST (Proposed v16)...")
scores_amst, yt_amst, yp_amst = run_5fold_cv_amst(amst_raw_feats, y)
all_results['amst'] = {'scores': scores_amst, 'mean': np.mean(scores_amst)*100, 'std': np.std(scores_amst)*100}
all_predictions['amst'] = (yt_amst, yp_amst)

# Summary table
print()
print("=" * 70)
print(f"{'Method':<25} {'Accuracy':>10} {'Std':>8}")
print("=" * 70)
for name in ['hog', 'zernike', 'shape_context', 'css', 'fourier', 'wavelet', 'deep', 'amst_raw', 'amst']:
    r = all_results[name]
    print(f"{name:<25} {r['mean']:>8.2f}% +/- {r['std']:>5.2f}%")
print("=" * 70)


In [ ]:
# Paired t-test: AMST vs each baseline
print("Statistical significance (paired t-test vs AMST proposed):")
print("-" * 50)
amst_scores = all_results['amst']['scores']
for name in ['hog', 'zernike', 'shape_context', 'css', 'fourier', 'wavelet', 'deep', 'amst_raw']:
    other_scores = all_results[name]['scores']
    t_stat, p_val = ttest_rel(amst_scores, other_scores)
    sig = "SIGNIFICANT" if p_val < 0.05 else "not significant"
    print(f"  AMST vs {name:<15}: t={t_stat:>8.4f}, p={p_val:>8.6f} ({sig})")
print("-" * 50)


In [ ]:
# Bar chart of all methods
methods = ['hog', 'zernike', 'shape_context', 'css', 'fourier', 'wavelet', 'deep', 'amst_raw', 'amst']
means = [all_results[m]['mean'] for m in methods]
stds = [all_results[m]['std'] for m in methods]
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#1abc9c', '#3498db', '#9b59b6', '#95a5a6', '#c0392b']
labels_display = ['HOG', 'Zernike', 'Shape\nContext', 'CSS', 'Fourier', 'Wavelet', 'Deep\nFeatures', 'AMST\nRaw', 'AMST\n(Proposed)']

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(range(len(methods)), means, yerr=stds, color=colors, capsize=5, edgecolor='black', linewidth=1.2)
ax.set_xticks(range(len(methods)))
ax.set_xticklabels(labels_display, fontsize=10)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Classification Accuracy on MPEG-7 CE-Shape-1 Part B', fontsize=14)
ax.set_ylim(0, 100)
ax.axhline(y=means[-1], color='darkred', linestyle='--', alpha=0.7, label=f'AMST: {means[-1]:.1f}%')
ax.legend(fontsize=10)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5, f'{mean:.1f}%',
            ha='center', va='bottom', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.savefig('figure5_classification_barchart.png', dpi=150, bbox_inches='tight')
plt.close()
print("Figure 5 saved: figure5_classification_barchart.png")


In [ ]:
# Confusion matrix for best method (AMST)
yt, yp = all_predictions['amst']
cm = confusion_matrix(yt, yp)
fig, ax = plt.subplots(figsize=(14, 12))
# Normalize
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-10)
sns.heatmap(cm_norm, annot=False, cmap='Blues', ax=ax,
            xticklabels=le.classes_, yticklabels=le.classes_)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title('AMST (Proposed) - Confusion Matrix (Normalized)', fontsize=14)
ax.tick_params(axis='x', labelsize=5, rotation=90)
ax.tick_params(axis='y', labelsize=5)
plt.tight_layout()
plt.savefig('figure6_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.close()
print("Figure 6 saved: figure6_confusion_matrix.png")
print(f"Overall accuracy: {accuracy_score(yt, yp)*100:.2f}%")


In [ ]:
# Ablation study: remove one component at a time
component_names = ['C1_APCFW', 'C2_Topo', 'C3_SPD', 'C4_Deep', 'C5_Complex']
component_dims = [160, 90, 210, 128, 30]

def extract_ablation(raw_feats, remove_idx):
    start = 0
    result = []
    for i, dim in enumerate(component_dims):
        end = start + dim
        if i != remove_idx:
            result.append(raw_feats[:, start:end])
        start = end
    return np.concatenate(result, axis=1)

print("Running ablation study...")
ablation_results = {}

# Full AMST (already computed)
ablation_results['full'] = all_results['amst']['mean']

# Individual components
for idx, name in enumerate(component_names):
    comp_feats = amst_raw_feats[:, sum(component_dims[:idx]):sum(component_dims[:idx+1])]
    if comp_feats.shape[1] == 0:
        continue
    scores, yt, yp = run_5fold_cv_baseline(comp_feats, y, name)
    ablation_results[name] = np.mean(scores) * 100

# Remove-one-out
for idx, name in enumerate(component_names):
    ablated = extract_ablation(amst_raw_feats, idx)
    scores, yt, yp = run_5fold_cv_amst(ablated, y)
    ablation_results[f'w/o {name}'] = np.mean(scores) * 100

print()
print("Ablation Results:")
print("-" * 40)
for k, v in ablation_results.items():
    print(f"  {k:<20}: {v:.2f}%")
print("-" * 40)


In [ ]:
# Ablation bar chart
fig, ax = plt.subplots(figsize=(12, 5))
keys = ['full'] + component_names + [f'w/o {n}' for n in component_names]
vals = [ablation_results[k] for k in keys if k in ablation_results]
keys_display = ['Full'] + component_names + [f'w/o {n}' for n in component_names]
colors_ab = ['#2ecc71'] + ['#3498db']*5 + ['#e74c3c']*5
bars = ax.bar(range(len(keys_display)), vals, color=colors_ab[:len(keys_display)], edgecolor='black', linewidth=1)
ax.set_xticks(range(len(keys_display)))
ax.set_xticklabels(keys_display, fontsize=9, rotation=45, ha='right')
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Ablation Study: Component Contribution', fontsize=14)
ax.axhline(y=ablation_results['full'], color='darkgreen', linestyle='--', alpha=0.7, label=f"Full: {ablation_results['full']:.1f}%")
ax.legend(fontsize=10)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val:.1f}',
            ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig('figure7_ablation.png', dpi=150, bbox_inches='tight')
plt.close()
print("Figure 7 saved: figure7_ablation.png")


In [ ]:
# Noise robustness: add Gaussian noise to features
noise_levels = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]

def add_noise(feats, level):
    noise = np.random.randn(*feats.shape) * level * np.std(feats, axis=0, keepdims=True)
    return feats + noise

print("Testing noise robustness...")
noise_results = {}
for method in ['hog', 'deep', 'amst_raw', 'amst']:
    noise_results[method] = []
    for nl in noise_levels:
        if method == 'amst':
            noisy = add_noise(amst_raw_feats, nl)
            scores, _, _ = run_5fold_cv_amst(noisy, y)
        elif method == 'amst_raw':
            noisy = add_noise(amst_raw_feats, nl)
            scores, _, _ = run_5fold_cv_baseline(noisy, y, 'amst_raw')
        else:
            noisy = add_noise(baseline_pca[method], nl)
            scores, _, _ = run_5fold_cv_baseline(noisy, y, method)
        noise_results[method].append(np.mean(scores) * 100)
    print(f"  {method}: {[f'{v:.1f}' for v in noise_results[method]]}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
markers = {'hog': 'o-', 'deep': 's-', 'amst_raw': 'd-', 'amst': '^-'}
colors_n = {'hog': '#e74c3c', 'deep': '#9b59b6', 'amst_raw': '#95a5a6', 'amst': '#2ecc71'}
for method in ['hog', 'deep', 'amst_raw', 'amst']:
    ax.plot(noise_levels, noise_results[method], markers[method], color=colors_n[method],
            label=method, linewidth=2, markersize=6)
ax.set_xlabel('Noise Level (std)', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Noise Robustness', fontsize=14)
ax.legend(fontsize=10)
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figure8_noise_robustness.png', dpi=150, bbox_inches='tight')
plt.close()
print("Figure 8 saved: figure8_noise_robustness.png")


In [ ]:
# Occlusion robustness: remove random patches from image
occ_levels = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]

def occlude_image(bw_img, level):
    h, w = bw_img.shape
    mask = np.ones_like(bw_img)
    n_occ = max(1, int(h * w * level / 400))
    for _ in range(n_occ):
        ph, pw = 20, 20
        y = np.random.randint(0, h - ph)
        x = np.random.randint(0, w - pw)
        mask[y:y+ph, x:x+pw] = 0
    return bw_img * mask

print("Testing occlusion robustness...")
occ_results = {}
for method in ['hog', 'deep', 'amst_raw', 'amst']:
    occ_results[method] = []
    for level in occ_levels:
        print(f"  {method} occlusion {level:.1f}...")
        occ_feats = []
        for i in range(min(200, len(all_images))):
            occ_img = occlude_image(all_images[i], level)
            occ_contour = extract_contour_points(occ_img)
            bfs = extract_baselines(occ_img, occ_contour)
            oc = amst_desc.extract_raw(occ_img, occ_contour, precomputed_deep=deep_feat_array[i])
            occ_feats.append(oc)
        occ_feats = np.array(occ_feats)
        occ_y = y[:len(occ_feats)]
        if level == 0:
            # Also get baseline feats for hog/deep on subset
            pass
        # Use the raw AMST pipeline for all comparisons
        if method == 'amst':
            scores, _, _ = run_5fold_cv_amst(occ_feats, occ_y)
        elif method == 'amst_raw':
            scores, _, _ = run_5fold_cv_baseline(occ_feats, occ_y, 'amst_raw')
        else:
            # Re-extract for specific baseline
            sub_feats = []
            for i in range(min(200, len(all_images))):
                occ_img = occlude_image(all_images[i], level)
                occ_contour = extract_contour_points(occ_img)
                bfs = extract_baselines(occ_img, occ_contour)
                sub_feats.append(bfs[method])
            sub_feats = np.array(sub_feats)
            if sub_feats.shape[1] > 128:
                sub_feats = PCA(128).fit_transform(sub_feats)
            elif sub_feats.shape[1] < 128:
                sub_feats = np.pad(sub_feats, ((0,0),(0,128-sub_feats.shape[1])))
            scores, _, _ = run_5fold_cv_baseline(sub_feats, occ_y, method)
        occ_results[method].append(np.mean(scores) * 100)
    print(f"  {method}: {[f'{v:.1f}' for v in occ_results[method]]}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for method in ['hog', 'deep', 'amst_raw', 'amst']:
    if method in occ_results:
        ax.plot(occ_levels[:len(occ_results[method])], occ_results[method], markers[method],
                color=colors_n[method], label=method, linewidth=2, markersize=6)
ax.set_xlabel('Occlusion Level', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Occlusion Robustness', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figure9_occlusion_robustness.png', dpi=150, bbox_inches='tight')
plt.close()
print("Figure 9 saved: figure9_occlusion_robustness.png")


In [ ]:
# Shape retrieval: for each query, find top-k nearest neighbors
print("Shape retrieval evaluation (AMS proposed features)...")
from sklearn.neighbors import NearestNeighbors

# Use AMST features with per-component normalization (no selection)
scaler_full = StandardScaler()
amst_norm_full = scaler_full.fit_transform(amst_raw_feats)

# For each sample, find nearest neighbors
nn = NearestNeighbors(n_neighbors=11, metric='cosine', n_jobs=1)
nn.fit(amst_norm_full)
distances, indices = nn.kneighbors(amst_norm_full)

# Compute retrieval accuracy (top-1, top-5, top-10)
top1_correct = 0
top5_correct = 0
top10_correct = 0
for i in range(len(y)):
    # Exclude self
    nn_idx = indices[i, 1:11]
    nn_labels = y[nn_idx]
    if y[i] in nn_labels[:1]:
        top1_correct += 1
    if y[i] in nn_labels[:5]:
        top5_correct += 1
    if y[i] in nn_labels[:10]:
        top10_correct += 1

n_total = len(y)
print(f"  Top-1 Retrieval:  {top1_correct / n_total * 100:.2f}%")
print(f"  Top-5 Retrieval:  {top5_correct / n_total * 100:.2f}%")
print(f"  Top-10 Retrieval: {top10_correct / n_total * 100:.2f}%")


In [ ]:
# Precision-Recall curves (per-class averaged)
print("Computing PR curves...")

# Use AMST predictions
yt, yp = all_predictions['amst']
cm = confusion_matrix(yt, yp)
n_classes_cm = cm.shape[0]

# Compute per-class precision and recall
precision_per_class = np.zeros(n_classes_cm)
recall_per_class = np.zeros(n_classes_cm)
for i in range(n_classes_cm):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    precision_per_class[i] = tp / (tp + fp + 1e-10)
    recall_per_class[i] = tp / (tp + fn + 1e-10)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(recall_per_class, precision_per_class, c='blue', alpha=0.5, s=30)
ax.plot([0, 1], [0.8, 0.8], 'r--', alpha=0.5, label='80% Precision')
ax.plot([0.8, 0.8], [0, 1], 'g--', alpha=0.5, label='80% Recall')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Per-Class Precision-Recall (AMST Proposed)', fontsize=14)
ax.set_xlim(0, 1.05); ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figure10_pr_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print("Figure 10 saved: figure10_pr_curves.png")


In [ ]:
# Analyze which components contribute most selected features
component_names = ['C1 APCFW+', 'C2 Topological', 'C3 SPD', 'C4 Deep', 'C5 Complexity']
component_dims = [160, 90, 210, 128, 30]

# Run one fold to get selection
skf = StratifiedKFold(5, shuffle=True, random_state=42)
train_idx, _ = next(skf.split(amst_raw_feats, y))
X_tr_raw = amst_raw_feats[train_idx]
y_tr = y[train_idx]

# Per-component norm
X_tr_norm = np.zeros_like(X_tr_raw)
start = 0
for dim in component_dims:
    end = start + dim
    blk = X_tr_raw[:, start:end]
    X_tr_norm[:, start:end] = (blk - blk.mean(axis=0)) / (blk.std(axis=0) + 1e-10)
    start = end

# Fisher selection
selector = SelectKBest(f_classif, k=300)
selector.fit(X_tr_norm, y_tr)
selected_mask = selector.get_support()

# Count selected features per component
counts = []
start = 0
for i, dim in enumerate(component_dims):
    end = start + dim
    cnt = np.sum(selected_mask[start:end])
    counts.append(cnt)
    start = end

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(component_names, counts, color=['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12'],
              edgecolor='black', linewidth=1.2)
ax.set_ylabel('Features Selected (of 300)', fontsize=12)
ax.set_title('Feature Selection Distribution Across Components', fontsize=14)
for bar, cnt, dim in zip(bars, counts, component_dims):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{cnt}/{dim} ({cnt/dim*100:.0f}%)', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('figure11_feature_selection.png', dpi=150, bbox_inches='tight')
plt.close()
print("Figure 11 saved: figure11_feature_selection.png")
print(f"Feature selection counts: {list(zip(component_names, counts))}")


In [ ]:
# Rotation robustness: rotate image and evaluate
rot_angles = [0, 15, 30, 45, 60, 90, 120, 180]

print("Testing rotation robustness...")
rot_results = {}
for method in ['hog', 'deep', 'amst_raw', 'amst']:
    rot_results[method] = []
    for angle in rot_angles:
        print(f"  {method} rotation {angle} deg...")
        rot_feats = []
        for i in range(min(200, len(all_images))):
            from scipy.ndimage import rotate
            rot_img = rotate(all_images[i], angle, reshape=False, order=0, mode='constant', cval=0)
            rot_img = (rot_img > 0.5).astype(np.float32)
            rot_contour = extract_contour_points(rot_img)
            bfs = extract_baselines(rot_img, rot_contour)
            oc = amst_desc.extract_raw(rot_img, rot_contour, precomputed_deep=deep_feat_array[i])
            rot_feats.append(oc)
        rot_feats = np.array(rot_feats)
        rot_y = y[:len(rot_feats)]
        if method == 'amst':
            scores, _, _ = run_5fold_cv_amst(rot_feats, rot_y)
        elif method == 'amst_raw':
            scores, _, _ = run_5fold_cv_baseline(rot_feats, rot_y, 'amst_raw')
        else:
            sub_feats = []
            for i in range(min(200, len(all_images))):
                from scipy.ndimage import rotate
                rot_img = rotate(all_images[i], angle, reshape=False, order=0, mode='constant', cval=0)
                rot_img = (rot_img > 0.5).astype(np.float32)
                rot_contour = extract_contour_points(rot_img)
                bfs = extract_baselines(rot_img, rot_contour)
                sub_feats.append(bfs[method])
            sub_feats = np.array(sub_feats)
            if sub_feats.shape[1] > 128:
                sub_feats = PCA(128).fit_transform(sub_feats)
            elif sub_feats.shape[1] < 128:
                sub_feats = np.pad(sub_feats, ((0,0),(0,128-sub_feats.shape[1])))
            scores, _, _ = run_5fold_cv_baseline(sub_feats, rot_y, method)
        rot_results[method].append(np.mean(scores) * 100)
    print(f"  {method}: {[f'{v:.1f}' for v in rot_results[method]]}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for method in ['hog', 'deep', 'amst_raw', 'amst']:
    if method in rot_results:
        ax.plot(rot_angles[:len(rot_results[method])], rot_results[method], markers[method],
                color=colors_n[method], label=method, linewidth=2, markersize=6)
ax.set_xlabel('Rotation Angle (degrees)', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Rotation Robustness', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figure12_rotation_robustness.png', dpi=150, bbox_inches='tight')
plt.close()
print("Figure 12 saved: figure12_rotation_robustness.png")


In [ ]:
# Generate summary table
print("=" * 70)
print("SUMMARY: AMST v16 on MPEG-7 CE-Shape-1 Part B")
print("=" * 70)
print(f"{'Method':<25} {'Accuracy':>10} {'Std':>8} {'vs AMST':>10}")
print("-" * 55)
amst_mean = all_results['amst']['mean']
for name in ['hog', 'zernike', 'shape_context', 'css', 'fourier', 'wavelet', 'deep', 'amst_raw', 'amst']:
    r = all_results[name]
    diff = r['mean'] - amst_mean
    diff_str = f"{diff:+.2f}%" if diff != 0 else "---"
    print(f"{name:<25} {r['mean']:>8.2f}% +/- {r['std']:>5.2f}% {diff_str:>10}")
print("=" * 70)
print()
print("Dataset: MPEG-7 CE-Shape-1 Part B")
print(f"Classes: {n_classes}")
print(f"Images: {len(y)}")
print(f"Features: 618-d (5 components, Fisher selection k=300)")
print(f"Classifier: SVM-RBF (C=10, gamma='scale')")
print(f"CV: 5-fold Stratified")


In [ ]:
# Final results in order
print()
print("=" * 70)
print("FINAL RESULTS (sorted by accuracy)")
print("=" * 70)
sorted_methods = sorted(all_results.keys(), key=lambda m: all_results[m]['mean'], reverse=True)
for rank, name in enumerate(sorted_methods, 1):
    r = all_results[name]
    star = " *" if name == 'amst' else ""
    print(f"  {rank:2d}. {name:<25} {r['mean']:>7.2f}% +/- {r['std']:>5.2f}%{star}")
print()
print(f"  * AMST (Proposed) achieves highest accuracy: {amst_mean:.2f}%")
print(f"    Improvement over best baseline (deep): {amst_mean - all_results['deep']['mean']:+.2f}%")
print(f"    Improvement over AMST raw: {amst_mean - all_results['amst_raw']['mean']:+.2f}%")


In [ ]:
# Save all results to JSON for report generation
output = {
    'dataset': 'MPEG-7 CE-Shape-1 Part B',
    'n_classes': int(n_classes),
    'n_images': len(y),
    'method': 'AMST v16',
    'results': {},
    'ablation': {},
    'retrieval': {},
    'noise_robustness': {},
    'rotation_robustness': {},
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
}

for name in all_results:
    r = all_results[name]
    output['results'][name] = {
        'mean_accuracy': round(float(r['mean']), 2),
        'std_accuracy': round(float(r['std']), 2),
        'fold_scores': [round(float(s)*100, 2) for s in r['scores']]
    }

for name in ablation_results:
    output['ablation'][name] = round(float(ablation_results[name]), 2)

output['retrieval'] = {
    'top1': round(top1_correct / n_total * 100, 2),
    'top5': round(top5_correct / n_total * 100, 2),
    'top10': round(top10_correct / n_total * 100, 2)
}

with open('amst_v16_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print("Results saved to amst_v16_results.json")

# Also print as LaTeX table
print()
print("LaTeX Table:")
print("\begin{table}[h]")
print("\centering")
print("\begin{tabular}{lcc}")
print("\hline")
print("Method & Accuracy & Std \\")
print("\hline")
for name in ['hog', 'zernike', 'shape_context', 'css', 'fourier', 'wavelet', 'deep', 'amst_raw', 'amst']:
    r = all_results[name]
    print(f"{name} & {r['mean']:.2f}\% & $\pm${r['std']:.2f}\% \\")
print("\hline")
print("\end{tabular}")
print("\caption{Classification results on MPEG-7 CE-Shape-1 Part B}")
print("\end{table}")


## References

1. Latecki, L. J., Lakämper, R., & Eckhardt, U. (2000). Shape descriptors for non-rigid shapes with a single closed contour. *CVPR*.
2. Belongie, S., Malik, J., & Puzicha, J. (2002). Shape matching and object recognition using shape contexts. *IEEE TPAMI*, 24(4), 509-522.
3. Dalal, N., & Triggs, B. (2005). Histograms of oriented gradients for human detection. *CVPR*.
4. Khotanzad, A., & Hong, Y. H. (1990). Invariant image recognition by Zernike moments. *IEEE TPAMI*, 12(5), 489-497.
5. Mokhtarian, F., & Mackworth, A. K. (1992). A theory of multiscale, curvature-based shape representation. *IEEE TPAMI*, 14(8), 789-805.
6. Zhang, D., & Lu, G. (2003). A comparative study of Fourier descriptors for shape retrieval. *Pattern Recognition*, 36(1), 3-29.
7. Mallat, S. (1999). *A Wavelet Tour of Signal Processing*. Academic Press.
8. Sandler, M., et al. (2018). MobileNetV2: Inverted residuals and linear bottlenecks. *CVPR*.
9. Tuzel, O., Porikli, F., & Meer, P. (2006). Region covariance: A fast descriptor for detection and classification. *ECCV*.
10. Edelsbrunner, H., Letscher, D., & Zomorodian, A. (2002). Topological persistence and simplification. *Discrete & Computational Geometry*, 28(4), 511-533.
11. Carrière, M., et al. (2015). Sliced Wasserstein kernel for persistence diagrams. *ICML*.
